# 제2장: 글로벌 AI 거버넌스 프레임워크

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch02.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

### 글로벌 AI 거버넌스 프레임워크 비교 분석

## 국제기구 및 표준화 기관의 프레임워크

### OECD AI 원칙(OECD AI Principles)

### UNESCO AI 윤리 권고안

### ISO/IEC 42001 AI 관리체계 표준

### IEEE 7000 시리즈

### NIST AI 리스크관리 프레임워크(AI RMF)

In [ ]:
"""NIST AI RMF Compliance Auto-Assessment Tool"""
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum
import json
from datetime import datetime

class ComplianceLevel(Enum):
    NOT_STARTED = 0
    PARTIAL = 1
    SUBSTANTIAL = 2
    FULL = 3

class RMFFunction(Enum):
    GOVERN = "Govern"
    MAP = "Map"
    MEASURE = "Measure"
    MANAGE = "Manage"

@dataclass
class SubCategory:
    id: str
    description: str
    compliance: ComplianceLevel = ComplianceLevel.NOT_STARTED
    evidence: str = ""
    action_plan: str = ""

@dataclass
class Category:
    id: str
    name: str
    function: RMFFunction
    subcategories: List[SubCategory] = field(
        default_factory=list
    )

    @property
    def compliance_score(self) -> float:
        if not self.subcategories:
            return 0.0
        total = sum(
            sc.compliance.value
            for sc in self.subcategories
        )
        return total / (
            len(self.subcategories)
            * ComplianceLevel.FULL.value
        ) * 100

class NISTAIRMFAssessment:
    """NIST AI RMF compliance assessment framework."""

    def __init__(self, org_name: str, system_name: str):
        self.org_name = org_name
        self.system_name = system_name
        self.assessment_date = datetime.now()
        self.categories: Dict[str, Category] = {}
        self._initialize_framework()

    def _initialize_framework(self):
        """Initialize all RMF categories and subcategories."""
        govern_cats = {
            "GV-1": ("Legal & Regulatory",
                      ["GV-1.1", "GV-1.2"]),
            "GV-2": ("Accountability Structure",
                      ["GV-2.1", "GV-2.2"]),
            "GV-3": ("Workforce Diversity",
                      ["GV-3.1", "GV-3.2"]),
            "GV-4": ("Org Competency",
                      ["GV-4.1", "GV-4.2", "GV-4.3"]),
            "GV-5": ("Engagement Mechanisms",
                      ["GV-5.1", "GV-5.2"]),
            "GV-6": ("Policies Documented",
                      ["GV-6.1", "GV-6.2"]),
        }
        for cat_id, (name, subs) in govern_cats.items():
            self.categories[cat_id] = Category(
                id=cat_id,
                name=name,
                function=RMFFunction.GOVERN,
                subcategories=[
                    SubCategory(id=s, description=f"{name} - {s}")
                    for s in subs
                ],
            )

        map_cats = {
            "MP-1": ("Intended Purpose", ["MP-1.1", "MP-1.2"]),
            "MP-2": ("Scientific Understanding",
                      ["MP-2.1", "MP-2.2"]),
            "MP-3": ("Risk Identification",
                      ["MP-3.1", "MP-3.2"]),
            "MP-4": ("Risk Prioritization", ["MP-4.1"]),
            "MP-5": ("Stakeholder Engagement",
                      ["MP-5.1", "MP-5.2"]),
        }
        for cat_id, (name, subs) in map_cats.items():
            self.categories[cat_id] = Category(
                id=cat_id,
                name=name,
                function=RMFFunction.MAP,
                subcategories=[
                    SubCategory(id=s, description=f"{name} - {s}")
                    for s in subs
                ],
            )

        measure_cats = {
            "ME-1": ("Metrics Selection", ["ME-1.1", "ME-1.2"]),
            "ME-2": ("Testing & Analysis", ["ME-2.1", "ME-2.2"]),
            "ME-3": ("Continuous Monitoring",
                      ["ME-3.1", "ME-3.2"]),
            "ME-4": ("Measurement Validation", ["ME-4.1"]),
        }
        for cat_id, (name, subs) in measure_cats.items():
            self.categories[cat_id] = Category(
                id=cat_id,
                name=name,
                function=RMFFunction.MEASURE,
                subcategories=[
                    SubCategory(id=s, description=f"{name} - {s}")
                    for s in subs
                ],
            )

        manage_cats = {
            "MG-1": ("Response Planning", ["MG-1.1", "MG-1.2"]),
            "MG-2": ("Response Implementation",
                      ["MG-2.1", "MG-2.2"]),
            "MG-3": ("Appeal Mechanisms", ["MG-3.1"]),
            "MG-4": ("Risk Communication", ["MG-4.1", "MG-4.2"]),
        }
        for cat_id, (name, subs) in manage_cats.items():
            self.categories[cat_id] = Category(
                id=cat_id,
                name=name,
                function=RMFFunction.MANAGE,
                subcategories=[
                    SubCategory(id=s, description=f"{name} - {s}")
                    for s in subs
                ],
            )

    def update_assessment(
        self,
        subcategory_id: str,
        compliance: ComplianceLevel,
        evidence: str = "",
        action_plan: str = "",
    ):
        """Update compliance status for a subcategory."""
        for cat in self.categories.values():
            for sc in cat.subcategories:
                if sc.id == subcategory_id:
                    sc.compliance = compliance
                    sc.evidence = evidence
                    sc.action_plan = action_plan
                    return
        raise ValueError(
            f"Subcategory {subcategory_id} not found"
        )

    def get_function_score(self, func: RMFFunction) -> float:
        """Calculate overall score for an RMF function."""
        func_cats = [
            c for c in self.categories.values()
            if c.function == func
        ]
        if not func_cats:
            return 0.0
        return sum(
            c.compliance_score for c in func_cats
        ) / len(func_cats)

    def get_overall_score(self) -> float:
        """Calculate overall compliance score."""
        return sum(
            self.get_function_score(f)
            for f in RMFFunction
        ) / len(RMFFunction)

    def generate_report(self) -> dict:
        """Generate compliance assessment report."""
        report = {
            "organization": self.org_name,
            "system": self.system_name,
            "assessment_date": self.assessment_date.isoformat(),
            "overall_score": round(
                self.get_overall_score(), 1
            ),
            "function_scores": {},
            "gap_analysis": [],
        }
        for func in RMFFunction:
            score = self.get_function_score(func)
            report["function_scores"][func.value] = round(
                score, 1
            )

        # Identify gaps (below substantial compliance)
        for cat in self.categories.values():
            for sc in cat.subcategories:
                if sc.compliance.value < ComplianceLevel.SUBSTANTIAL.value:
                    report["gap_analysis"].append({
                        "subcategory": sc.id,
                        "current_level": sc.compliance.name,
                        "description": sc.description,
                        "action_plan": sc.action_plan,
                    })
        return report


# Usage example
if __name__ == "__main__":
    assessment = NISTAIRMFAssessment(
        org_name="Example Corp Korea",
        system_name="AI Credit Scoring v2.1",
    )

    # Assess several subcategories
    assessment.update_assessment(
        "GV-1.1",
        ComplianceLevel.FULL,
        evidence="Regulatory tracking system deployed",
    )
    assessment.update_assessment(
        "GV-2.1",
        ComplianceLevel.SUBSTANTIAL,
        evidence="RACI matrix documented",
    )
    assessment.update_assessment(
        "MP-3.1",
        ComplianceLevel.PARTIAL,
        evidence="Initial risk register created",
        action_plan="Complete risk assessment by Q2",
    )
    assessment.update_assessment(
        "ME-1.1",
        ComplianceLevel.NOT_STARTED,
        action_plan="Define fairness metrics by Q1",
    )

    report = assessment.generate_report()
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print(f"\nOverall Score: {report['overall_score']}%")
    print(f"Gaps found: {len(report['gap_analysis'])}")

### 프레임워크 간 상호운용성 정렬

## 주요국 규제 동향

### 유럽연합: EU AI Act 심층 분석

### 미국: 연방 주도 자율 규제로의 전환

### 중국: 알고리즘 거버넌스와 콘텐츠 라벨링 강화

### 일본: 소프트 로(Soft Law) 중심의 접근

### 싱가포르: 실용주의적 거버넌스 프레임워크

## 국내 AI 규제 환경

### AI 기본법 심층 분석

### 기타 주요 규제

## 규제 수렴과 상호운용성

### 규제 격차 분석(Gap Analysis) 방법론

### 다국적 기업의 통합 거버넌스 전략

## 규제 대응 실무 도구

### 글로벌 AI 규제 비교 매트릭스

### 프레임워크 선택 가이드

### 규제 매핑 자동화 도구

In [ ]:
"""Multi-Jurisdiction AI Regulation Mapping Tool"""
from dataclasses import dataclass, field
from typing import Dict, List, Set, Optional
from enum import Enum
import json

class RiskLevel(Enum):
    MINIMAL = "minimal"
    LIMITED = "limited"
    HIGH = "high"
    UNACCEPTABLE = "unacceptable"

class Jurisdiction(Enum):
    EU = "EU"
    US = "US"
    CHINA = "China"
    KOREA = "Korea"
    SINGAPORE = "Singapore"
    JAPAN = "Japan"

class ComplianceStatus(Enum):
    NOT_ASSESSED = "not_assessed"
    NON_COMPLIANT = "non_compliant"
    PARTIAL = "partial"
    SUBSTANTIAL = "substantial"
    COMPLIANT = "compliant"

@dataclass
class RegulatoryRequirement:
    id: str
    jurisdiction: Jurisdiction
    description: str
    risk_level: RiskLevel
    deadline: str
    mandatory: bool = True

@dataclass
class AISystem:
    name: str
    description: str
    domain: str
    jurisdictions: Set[Jurisdiction] = field(
        default_factory=set
    )
    risk_classification: Dict[Jurisdiction, RiskLevel] = (
        field(default_factory=dict)
    )

class RegulatoryMapper:
    """Maps AI systems to applicable regulations."""

    def __init__(self):
        self.requirements: List[RegulatoryRequirement] = []
        self.ai_systems: List[AISystem] = []
        self._load_requirements()

    def _load_requirements(self):
        """Load regulatory requirements database."""
        # EU AI Act requirements
        eu_reqs = [
            RegulatoryRequirement(
                "EU-HIGH-001", Jurisdiction.EU,
                "Risk management system",
                RiskLevel.HIGH, "2026-08-02",
            ),
            RegulatoryRequirement(
                "EU-HIGH-002", Jurisdiction.EU,
                "Data governance measures",
                RiskLevel.HIGH, "2026-08-02",
            ),
            RegulatoryRequirement(
                "EU-HIGH-003", Jurisdiction.EU,
                "Technical documentation",
                RiskLevel.HIGH, "2026-08-02",
            ),
            RegulatoryRequirement(
                "EU-HIGH-004", Jurisdiction.EU,
                "Human oversight mechanism",
                RiskLevel.HIGH, "2026-08-02",
            ),
            RegulatoryRequirement(
                "EU-HIGH-005", Jurisdiction.EU,
                "Conformity assessment",
                RiskLevel.HIGH, "2026-08-02",
            ),
            RegulatoryRequirement(
                "EU-GPAI-001", Jurisdiction.EU,
                "Technical documentation for GPAI",
                RiskLevel.LIMITED, "2025-08-02",
            ),
            RegulatoryRequirement(
                "EU-GPAI-002", Jurisdiction.EU,
                "Copyright compliance policy",
                RiskLevel.LIMITED, "2025-08-02",
            ),
            RegulatoryRequirement(
                "EU-TRANS-001", Jurisdiction.EU,
                "AI interaction transparency",
                RiskLevel.LIMITED, "2026-08-02",
            ),
        ]
        self.requirements.extend(eu_reqs)

        # Korea AI Basic Act requirements
        kr_reqs = [
            RegulatoryRequirement(
                "KR-HIGH-001", Jurisdiction.KOREA,
                "Impact assessment for high-risk AI",
                RiskLevel.HIGH, "2026-07-01",
            ),
            RegulatoryRequirement(
                "KR-HIGH-002", Jurisdiction.KOREA,
                "Explanation obligation",
                RiskLevel.HIGH, "2026-07-01",
            ),
            RegulatoryRequirement(
                "KR-HIGH-003", Jurisdiction.KOREA,
                "Safety measures for high-risk AI",
                RiskLevel.HIGH, "2026-07-01",
            ),
            RegulatoryRequirement(
                "KR-GEN-001", Jurisdiction.KOREA,
                "Generative AI content labeling",
                RiskLevel.LIMITED, "2026-07-01",
            ),
        ]
        self.requirements.extend(kr_reqs)

        # China requirements
        cn_reqs = [
            RegulatoryRequirement(
                "CN-LABEL-001", Jurisdiction.CHINA,
                "AI content explicit labeling",
                RiskLevel.LIMITED, "2025-09-01",
            ),
            RegulatoryRequirement(
                "CN-LABEL-002", Jurisdiction.CHINA,
                "AI content implicit labeling (watermark)",
                RiskLevel.LIMITED, "2025-09-01",
            ),
            RegulatoryRequirement(
                "CN-ALGO-001", Jurisdiction.CHINA,
                "Algorithm registration filing",
                RiskLevel.LIMITED, "2022-03-01",
            ),
        ]
        self.requirements.extend(cn_reqs)

    def register_system(self, system: AISystem):
        """Register an AI system for regulatory mapping."""
        self.ai_systems.append(system)

    def get_applicable_requirements(
        self, system: AISystem
    ) -> List[RegulatoryRequirement]:
        """Get all requirements applicable to a system."""
        applicable = []
        for req in self.requirements:
            if req.jurisdiction not in system.jurisdictions:
                continue
            sys_risk = system.risk_classification.get(
                req.jurisdiction, RiskLevel.MINIMAL
            )
            risk_order = [
                RiskLevel.MINIMAL, RiskLevel.LIMITED,
                RiskLevel.HIGH, RiskLevel.UNACCEPTABLE,
            ]
            if (risk_order.index(sys_risk)
                    >= risk_order.index(req.risk_level)):
                applicable.append(req)
        return applicable

    def gap_analysis(
        self,
        system: AISystem,
        current_status: Dict[str, ComplianceStatus],
    ) -> dict:
        """Perform gap analysis for a system."""
        applicable = self.get_applicable_requirements(system)
        gaps = []
        compliant_count = 0

        for req in applicable:
            status = current_status.get(
                req.id, ComplianceStatus.NOT_ASSESSED
            )
            if status in (
                ComplianceStatus.COMPLIANT,
                ComplianceStatus.SUBSTANTIAL,
            ):
                compliant_count += 1
            else:
                gaps.append({
                    "requirement_id": req.id,
                    "jurisdiction": req.jurisdiction.value,
                    "description": req.description,
                    "deadline": req.deadline,
                    "current_status": status.value,
                    "mandatory": req.mandatory,
                })

        total = len(applicable)
        return {
            "system": system.name,
            "total_requirements": total,
            "compliant": compliant_count,
            "compliance_rate": (
                round(compliant_count / total * 100, 1)
                if total > 0 else 0
            ),
            "gaps": sorted(
                gaps, key=lambda x: x["deadline"]
            ),
        }


# Usage example
if __name__ == "__main__":
    mapper = RegulatoryMapper()

    # Register a credit scoring AI system
    credit_ai = AISystem(
        name="AI Credit Scoring System",
        description="ML-based credit evaluation",
        domain="finance",
        jurisdictions={
            Jurisdiction.EU, Jurisdiction.KOREA,
        },
        risk_classification={
            Jurisdiction.EU: RiskLevel.HIGH,
            Jurisdiction.KOREA: RiskLevel.HIGH,
        },
    )
    mapper.register_system(credit_ai)

    # Current compliance status
    status = {
        "EU-HIGH-001": ComplianceStatus.SUBSTANTIAL,
        "EU-HIGH-002": ComplianceStatus.PARTIAL,
        "EU-HIGH-003": ComplianceStatus.PARTIAL,
        "KR-HIGH-001": ComplianceStatus.NOT_ASSESSED,
    }

    result = mapper.gap_analysis(credit_ai, status)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print(f"\nCompliance Rate: {result['compliance_rate']}%")
    print(f"Gaps to address: {len(result['gaps'])}")

### 규제 변화 추적 체계

### 규제 대응 체크리스트

### 제2장 요약